#### **Simple Prompt Template**

**📌 Best For:**
- Basic Q&A
- Summarization
- Simple generation tasks
- When you only need 1 formatted string

**🚫 Avoid When:**
- You need chat-style messages
- You need structured system/user separation

In [1]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    template='''Write 5 sentence summary about {Topic}''',
)


print(template.invoke('Cricket'))

c:\Users\maham\Desktop\Git\aii\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


text='Write 5 sentence summary about Cricket'


In [2]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    template='''Write {n} sentence summary about {topic}''',
    input_variables=['n', 'topic'],
    validate_template=True
)

n = 5
print(template.invoke({'n':n,'topic':'Cricket'}))
print(len(template.invoke({'n':n,'topic':'Cricket'})))

text='Write 5 sentence summary about Cricket'


TypeError: object of type 'StringPromptValue' has no len()

**Save Template**

In [ ]:
# template.save('/prompts_s/template.json')
# template.save('/prompts_s/template.yaml')
template.save("temp.json")

**Load Template**

In [ ]:
from langchain_core.prompts import load_prompt

tem = load_prompt("./temp.json")
print(tem)
print(len(tem))

#### **Chat Prompt Template**

**Use When:**
- Using chat models (GPT-4, Claude, Gemini)
- You need system instructions
- You need structured conversations

📌 This is the default choice for production LLM apps.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "Explain {topic}.")
])

print(prompt)

#### **🔁 3. MessagesPlaceholder**

**Production Use Case:**
- Chatbots
- Agents
- RAG with memory
- Multi-turn systems

**If your app has:**
- stored messages
- session-based history
- conversation replay

👉 You need MessagesPlaceholder.
✅ Use When: You have conversation memory

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# Define LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Define parser
parser = StrOutputParser()

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("chat_history"),
    ("user", "{input}")
])

# Create chain
chain = prompt | llm | parser

# Example chat history
chat_history = [
    ("user", "Hello!"),
    ("assistant", "Hi! How can I help you today?")
]

# Invoke the chain with required inputs
response = chain.invoke({
    "input": "Can you explain what LangChain is?",
    "chat_history": chat_history
})

print(response)

#### **🧩 4. FewShotPromptTemplate**
✅ Use When: You need better reasoning or format control

**Production Use Case:**
- Classification
- Extraction
- Format-sensitive outputs
- Reducing hallucination
**Example:** show 3 correct examples → model copies pattern.

🚫 **Avoid When:**
- You're already using structured output parsers
- You don’t actually need examples

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 1. Create example template
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}"
)

# 2. Define few-shot examples
examples = [
    {"input": "2 + 2", "output": "4"},
    {"input": "3 + 5", "output": "8"},
]

# 3. Create FewShotPromptTemplate
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Solve the following math problems:",
    suffix="Input: {input}\nOutput:",
    input_variables=["input"],
)

# 4. Create LLM + parser
llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

# 5. Build chain
chain = few_shot_prompt | llm | parser

# 6. Invoke
response = chain.invoke({"input": "10 + 15"})
print(response)

#### **🔄 6. Partial Variables**

✅**Use When:**
- Some variables are static
- Injecting environment config
- Multi-tenant systems
- Very common in SaaS apps.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 1️⃣ Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant for {company_name}."),
    ("user", "{question}")
]).partial(company_name="OpenAI")  # <-- Pre-fill variable

# 2️⃣ Initialize model + parser
llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

# 3️⃣ Build chain
chain = prompt | llm | parser

# 4️⃣ Invoke (notice we only pass 'question')
response = chain.invoke({
    "question": "What does this company do?"
})

print(response)

**🔎 What .partial() Does**

It pre-fills one or more variables in the prompt so you don’t have to pass them every time.

Without .partial() you'd need:
```py
chain.invoke({
    "company_name": "OpenAI",
    "question": "What does this company do?"
})
```
With .partial(), company_name is permanently set inside the template.

**🧠 When to Use .partial()**

**Use it when:**
- A variable never changes (like company name, tone, language)
- You want cleaner invoke() calls
- You are building reusable chains

# 🎯 What To Use In Production (Simple Rule)

| Situation | Use |
|-----------|-----|
| Simple one-off prompt | `PromptTemplate` |
| Chat model | `ChatPromptTemplate` |
| Conversation memory | `MessagesPlaceholder` |
| Format control | `FewShotPromptTemplate` |
| JSON/API extraction | `with_structured_output()` |
| Static injected variables | `.partial()` |
| Complex flows | LCEL composition |

---

#### 🏗 **Real Production Patterns**

##### 🔹 RAG App
- `ChatPromptTemplate`  
- `MessagesPlaceholder`  
- Context injection  
- Structured output (optional)  

##### 🔹 AI API Backend
- `ChatPromptTemplate`  
- Structured output via Pydantic  
- Strict validation  

##### 🔹 Autonomous Agent
- `ChatPromptTemplate`  
- Tool calling  
- Structured output  

---

##### ⚠️ **Why There Are So Many Prompt Classes?**

Because LangChain evolved:

| Old Era | New Era |
|---------|---------|
| Chains | Runnables |
| LLMChain | `prompt | model` |
| Manual parsing | Structured output |

> If you’re on v0.1.2+, ignore most legacy tutorials.

---

##### 🧭 **Decision Tree**

- **Am I using a chat model?**  
  → Yes → `ChatPromptTemplate`

- **Do I need memory?**  
  → Yes → `MessagesPlaceholder`

- **Do I need strict JSON?**  
  → Yes → `with_structured_output()`

- **Do I need examples?**  
  → Yes → `FewShotPromptTemplate`

> If none apply → keep it simple (`PromptTemplate`).

---

##### 🏆 **Production Advice (Important)**

- Keep prompts small  
- Avoid unnecessary few-shot  
- Prefer structured output over regex parsing  
- Keep system message stable  
- Version your prompts  

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, AIMessagePromptTemplate,ChatMessagePromptTemplate, FewShotPromptTemplate, 